# Task 4 — Valutazione dei classificatori manuali su `training.csv`

> 📄 Documentazione completa e motivazioni: [`docs/task4.md`](../docs/task4.md)

## 0. Setup e dati

Il **punto 4 della traccia** chiede di valutare i due classificatori costruiti a mano nel Task 2
(1R e Naïve Bayes) sul dataset reale `training.csv` (o un suo sottoinsieme), cercando di
**ottimizzarne le prestazioni**. A differenza del Task 2, qui lavoriamo su ~41.000 istanze
**fortemente sbilanciate**, vedremo quindi come i due modelli precedentemente addestrati sul dataset da sole 12 righe si comporteranno.

In [1]:
import pandas as pd
import numpy as np


In [2]:
training = pd.read_csv("../data/processed/training.csv", sep=";")
dataFrame_NB = training.copy() # copia del dataset originale così da non modificarlo
dataFrame_1R = training.copy() 

## 1. Inizializzazione variabili e metodi 

Nelle 2 celle seguenti riprendiamo la variabile **probabilita** , con alcune modifiche per gestire il valore unknown , e la funzione **predici_naive_bayes**, presenti in **task2_NaiveBayes.ipynb**.

In [3]:
#Tutte le probabilità condizionate calcolate con lo stimatore di Laplace.
#Rispetto al Task 2 'marital' acquista un quarto valore, 'unknown' (assente in manuale.csv ma
#presente in training.csv, 80 righe). 'unknown' è una categoria legittima del Bank Marketing,
#non un dato mancante. Avendo ora k=4 categorie, il denominatore di Laplace per 'marital' passa
#da Nc+3=9 a Nc+k=6+4=10: RICALCOLIAMO tutte le sue probabilità così che continuino a sommare a 1.
#(housing e loan avevano già 'unknown' tra i 3 valori, quindi restano con denominatore 9.)
#N.B.: il riscalamento del solo denominatore di 'marital' agisce in egual misura sulle due classi,
#quindi le predizioni del classificatore NON cambiano; cambia solo la validità della distribuzione.
probabilita = {
    "marital": {
        0: {"divorced": 2/13, "married": 8/13, "single": 1/13, "unknown": 1/13},
        1: {"divorced": 3/13, "married": 4/13, "single": 4/13, "unknown": 1/13},
    },
    "housing": {
        0: {"no": 3/9, "unknown": 1/9, "yes": 5/9},
        1: {"no": 3/9, "unknown": 2/9, "yes": 4/9},
    },
    "loan": {
        0: {"no": 6/9, "unknown": 1/9, "yes": 2/9},
        1: {"no": 6/9, "unknown": 2/9, "yes": 1/9},
    }
}

In [4]:
def predici_naive_bayes(row):
    # Probabilità a priori
    score_0 = 0.5
    score_1 = 0.5

    # marital
    score_0 *= probabilita["marital"][0][row["marital"]] 
    score_1 *= probabilita["marital"][1][row["marital"]]

    # housing
    score_0 *= probabilita["housing"][0][row["housing"]]
    score_1 *= probabilita["housing"][1][row["housing"]]

    # loan
    score_0 *= probabilita["loan"][0][row["loan"]]
    score_1 *= probabilita["loan"][1][row["loan"]]



    # Predizione finale 
    if (score_0 > score_1):
        prediction = 0
    else:
        prediction = 1
    return prediction, score_0, score_1

## 1.1 Naïve Bayes su `training.csv`

Valutiamo il Naïve Bayes (qui nella variante `GaussianNB` di scikit-learn, come controprova
sull'intero dataset). Un punto metodologico importante riguarda il **leakage**: la codifica dei
nominali con `OrdinalEncoder` viene **adattata (`fit`) solo sul training** e poi semplicemente
**applicata (`transform`)** al test, così che nessuna informazione del test influenzi
l'addestramento. Confrontiamo due scenari — **con** e **senza** l'attributo `duration` — per
misurare quanto questa variabile "drogata" dal leakage gonfi i risultati.

In [5]:
# Usa SOLO le colonne del modello Naive Bayes
training_nb = dataFrame_NB[["marital", "housing", "loan"]].copy()

training_nb["Predicted"] = training_nb.apply(
    lambda row: predici_naive_bayes(row)[0],
    axis=1
)

#chiamata alla funzione predici_naive_bayes , con le probabilità calcolate in precedenza
dataFrame_NB["Predicted"] = dataFrame_NB.apply(lambda row: predici_naive_bayes(row)[0],axis=1)
dataFrame_NB.head(10)

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y,Predicted
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0
5,45,services,married,basic.9y,unknown,no,no,telephone,may,mon,...,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0
6,59,admin.,married,professional.course,no,no,no,telephone,may,mon,...,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0
7,41,blue-collar,married,unknown,unknown,no,no,telephone,may,mon,...,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0
8,24,technician,single,professional.course,no,yes,no,telephone,may,mon,...,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,1
9,25,services,single,high.school,no,yes,no,telephone,may,mon,...,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,1


In [6]:
accuracy = (dataFrame_NB["y"] == dataFrame_NB["Predicted"]).mean()
tp = len(dataFrame_NB[(dataFrame_NB["y"] == 1) & (dataFrame_NB["Predicted"] == 1)])
tn = len(dataFrame_NB[(dataFrame_NB["y"] == 0) & (dataFrame_NB["Predicted"] == 0)])
fp = len(dataFrame_NB[(dataFrame_NB["y"] == 0) & (dataFrame_NB["Predicted"] == 1)])
fn = len(dataFrame_NB[(dataFrame_NB["y"] == 1) & (dataFrame_NB["Predicted"] == 0)])


confusion_matrix_df = pd.DataFrame(
    [[tn, fp],
     [fn, tp]],
    columns=["Predicted 0", "Predicted 1"],
    index=["Actual 0", "Actual 1"]
)
confusion_matrix_df

precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * (precision * recall) / (precision + recall)
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

Accuracy : 0.5971
Precision: 0.1296
Recall   : 0.4507
F1 Score : 0.2013


## 2. 1R manuale su `training.csv`

Riprendiamo ora il classificatore **1R** del Task 2. Sul `manuale.csv` 1R aveva scelto
l'attributo **`job`**, con la regola (una classe per ciascun valore osservato):

| valore di `job` | predizione |
|---|---|
| admin., retired, student, technician | 1 (`yes`) |
| blue-collar, unknown | 0 (`no`) |

Applichiamo **la stessa regola, invariata**, a `training.csv`.

### 2.1 Il problema dei valori mai visti

`manuale.csv` conteneva solo 6 dei 12 valori di `job` presenti in `training.csv`. I 6 valori
**mai visti in fase di addestramento** (`entrepreneur`, `housemaid`, `management`,
`self-employed`, `services`, `unemployed`) non hanno una regola associata: coerentemente con la
funzione `predici_1R` del Task 2 (`regole.get(val, 0)`), vengono predetti con il **default `0`**.
È una conseguenza diretta dell'aver *congelato* un modello appreso su pochissimi dati: lo
segnaliamo esplicitamente perché incide sul risultato.

In [7]:
modello_1R = {'attributo': 'job', 
              'regole': {'admin.': 1, 'blue-collar': 0, 'retired': 1, 'student': 1, 'technician': 1, 'unknown': 0},
              'soglia': None, 'errori': 2}

# Valori di 'job' presenti in training.csv ma mai visti su manuale.csv
valori_mai_visti = sorted(set(dataFrame_1R["job"].unique()) - set(modello_1R['regole'].keys()))
print("Valori di 'job' mai visti (predetti con default 0):")
print(valori_mai_visti)

Valori di 'job' mai visti (predetti con default 0):
['entrepreneur', 'housemaid', 'management', 'self-employed', 'services', 'unemployed']


In [8]:
def predici_1R(modello, istanza):
    attr = modello["attributo"]
    if modello["soglia"] is not None:                 # attributo numerico
        val = "high" if istanza[attr] > modello["soglia"] else "low"
    else:                                             # attributo nominale
        val = istanza[attr]
    return modello["regole"].get(val, 0)

In [9]:
dataFrame_1R["Predicted"] = dataFrame_1R.apply(lambda row: predici_1R(modello_1R, row),axis=1)
dataFrame_1R.head(10)

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y,Predicted
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,1
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0
5,45,services,married,basic.9y,unknown,no,no,telephone,may,mon,...,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0
6,59,admin.,married,professional.course,no,no,no,telephone,may,mon,...,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,1
7,41,blue-collar,married,unknown,unknown,no,no,telephone,may,mon,...,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0
8,24,technician,single,professional.course,no,yes,no,telephone,may,mon,...,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,1
9,25,services,single,high.school,no,yes,no,telephone,may,mon,...,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0


In [10]:
accuracy = (dataFrame_1R["y"] == dataFrame_1R["Predicted"]).mean()
tp = len(dataFrame_1R[(dataFrame_1R["y"] == 1) & (dataFrame_1R["Predicted"] == 1)])
tn = len(dataFrame_1R[(dataFrame_1R["y"] == 0) & (dataFrame_1R["Predicted"] == 0)])
fp = len(dataFrame_1R[(dataFrame_1R["y"] == 0) & (dataFrame_1R["Predicted"] == 1)])
fn = len(dataFrame_1R[(dataFrame_1R["y"] == 1) & (dataFrame_1R["Predicted"] == 0)])


confusion_matrix_df = pd.DataFrame(
    [[tn, fp],
     [fn, tp]],
    columns=["Predicted 0", "Predicted 1"],
    index=["Actual 0", "Actual 1"]
)
confusion_matrix_df

precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * (precision * recall) / (precision + recall)
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

Accuracy : 0.5432
Precision: 0.1413
Recall   : 0.6014
F1 Score : 0.2288


## 3. Confronto finale

Mettiamo a confronto i tre modelli sulle quattro metriche richieste. La lettura va fatta
**privilegiando recall e F1 sulla classe `yes`**, non l'accuratezza.

In [11]:
# Funzione di supporto: calcola le 4 metriche a partire dalle predizioni già ottenute.
# La classe positiva di riferimento è y = 1 (cliente che sottoscrive il deposito).
def metriche_da_predizioni(y_true, y_pred):
    tp = len(y_true[(y_true == 1) & (y_pred == 1)])
    tn = len(y_true[(y_true == 0) & (y_pred == 0)])
    fp = len(y_true[(y_true == 0) & (y_pred == 1)])
    fn = len(y_true[(y_true == 1) & (y_pred == 0)])
    accuracy  = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return accuracy, precision, recall, f1


# Classe reale (identica nei due DataFrame, sono copie dello stesso training.csv)
y = dataFrame_NB["y"]

# Baseline "sempre no": predice sempre la classe maggioritaria 0
baseline_pred = pd.Series(0, index=y.index)

# Tabella di confronto sulle quattro metriche richieste
confronto = pd.DataFrame(
    {
        "Baseline (sempre no)":                 metriche_da_predizioni(y, baseline_pred),
        "Naive Bayes (marital, housing, loan)": metriche_da_predizioni(y, dataFrame_NB["Predicted"]),
        "1R (job)":                             metriche_da_predizioni(y, dataFrame_1R["Predicted"]),
    },
    index=["accuracy", "precision", "recall", "f1"],
).T.round(4)

confronto

,accuracy,precision,recall,f1
Baseline (sempre no),0.8873,0.0000,0.0000,0.0000
"Naive Bayes (marital, housing, loan)",0.5971,0.1296,0.4507,0.2013
1R (job),0.5432,0.1413,0.6014,0.2288


### 3.1 Osservazioni sul confronto

La tabella mette a fuoco il punto centrale del task: **su un dataset così sbilanciato
l'accuratezza è una metrica fuorviante**, e vanno lette soprattutto **recall e F1** sulla
classe `yes`.

- **Baseline "sempre no".** Ottiene l'accuratezza più alta in assoluto (~88,7 %), semplicemente
  perché l'88,7 % delle istanze è davvero `no`. Ma ha **recall e F1 pari a 0**: non individua
  *nessun* cliente propenso. È il classificatore inutile per definizione, e serve proprio a
  smascherare l'accuratezza come metrica ingannevole.
- **Naïve Bayes e 1R sacrificano accuratezza per recall.** Entrambi scendono ben sotto il
  baseline in accuratezza (NB ~0,60, 1R ~0,54), ma in cambio recuperano una quota consistente di
  `yes`: il loro **recall è nettamente maggiore di 0**, e con esso l'F1. Pur essendo stati
  appresi su sole 12 righe, **generalizzano** abbastanza da catturare parte della classe rara.
- **1R vs Naïve Bayes.** 1R, con il solo attributo `job`, ottiene il **recall più alto**
  (~0,60) e di conseguenza l'**F1 migliore** (~0,23); il Naïve Bayes, con tre attributi, ha
  accuratezza un po' più alta ma recall inferiore (~0,45). In entrambi i casi la **precision
  resta molto bassa** (~0,13–0,14): i modelli segnalano *troppi* `yes`, generando molti falsi
  positivi.

### 3.2 Analisi critica dei risultati

Il risultato va interpretato alla luce del fatto che stiamo applicando **modelli congelati**:
regole e probabilità sono quelle apprese nel Task 2 sulle 12 righe di `manuale.csv`, e non
vengono ricalcolate su `training.csv`. Da qui derivano tre limiti strutturali.

**Distribuzione di addestramento diversa da quella di test.** `manuale.csv` era *bilanciato*
(6 `yes` / 6 `no`), mentre `training.csv` ha solo ~11 % di `yes`. I modelli hanno quindi imparato
una propensione al `yes` molto più alta di quella reale: questo spiega l'**altissimo numero di
falsi positivi** e la precision bassa. È il classico effetto di un *prior* di training non
rappresentativo della popolazione reale.

**Pochi attributi, tutti deboli.** Il Naïve Bayes usa solo `marital`, `housing`, `loan` — tre
variabili con potere predittivo modesto sul target. 1R, per costruzione, ne usa **uno solo**
(`job`). Nessuno dei due ha accesso agli attributi realmente discriminanti emersi nell'EDA del
Task 3 (es. `poutcome`, le variabili economiche). Da qui un'F1 strutturalmente bassa.

**Valori mai visti.** 1R non ha una regola per 6 dei 12 valori di `job` e li manda tutti sul
default `0`: una scelta prudente, ma che riduce ulteriormente il recall potenziale. È
l'evidenza diretta del rischio di **overfitting su un campione minuscolo**: il modello non ha
mai osservato metà delle categorie su cui ora deve decidere.

Nonostante ciò, il messaggio positivo è chiaro: **entrambi i modelli battono nettamente il
baseline su recall e F1**, cioè proprio sulle metriche che contano quando l'obiettivo di business
è *non perdere* i clienti propensi di una campagna di marketing.

### 3.3 Nota sull'ottimizzazione

La traccia chiede di **"ottimizzare le prestazioni"** dei classificatori. Avendo scelto di
valutarli *congelati* (così come appresi nel Task 2, senza riaddestrarli), il margine di
ottimizzazione disponibile in questo task è **strutturalmente limitato**: i parametri sono fissati
dalle 12 righe di `manuale.csv` e non dipendono da `training.csv`.

L'unico intervento di adattamento necessario è stato la **gestione del valore `unknown` di
`marital`** (assente in `manuale.csv` ma presente in `training.csv`): gli abbiamo assegnato una
probabilità tramite lo stesso **smoothing di Laplace** già usato nel Task 2, così da mantenere la
distribuzione valida senza alterare le predizioni. Senza questo accorgimento il Naïve Bayes
solleverebbe un `KeyError` sulle righe con `marital = unknown`.

Le leve di ottimizzazione vere e proprie — **riaddestrare** i modelli su `training.csv`,
**aggiungere attributi** più informativi, **discretizzare** le variabili numeriche, e soprattutto
**gestire lo sbilanciamento** delle classi (es. con pesi di classe o ricampionamento) —
richiedono di costruire e addestrare nuovi modelli, ed è esattamente l'oggetto del **Task 5**, dove
si realizzano pipeline complete con scikit-learn. In questo senso il Task 4 ha soprattutto un
valore **diagnostico**: mostra *perché* serviranno quelle tecniche, quantificando i limiti dei
classificatori manuali su un dataset grande e sbilanciato.